In [1]:
import requests
from pathlib import Path
import time

In [2]:
BASE_DIR = Path("/mnt/clips_home/llm_cleanup_new_code/JSON_NO_STRUCTURE_EXTRACTED_TXT")
OUTPUT_DIR = Path("/mnt/clips_home/llm_cleanup_new_code/NO_STRUCTURE_LLM_CLEANED")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


MODEL = "qwen2.5:7b-instruct"
OLLAMA_URL = "http://127.0.0.1:11435/api/chat" 
TIMEOUT = 1500


SYSTEM_PROMPT = """
You extract the representation body from planning documents. Your ONLY task is to identify the start and end of the representation body and COPY its textual content exactly.
The representation body is a text expressing views, objections, support, concerns or reasoning regarding the planning application.

The input may contain:
- administrative headers
- representation titles or labels (e.g. "Representation 002 Support")
- names, addresses, and contact details
- salutations and signatures
- an attached document containing the substantive representation
- appendices or other supplementary attachments
- the main representation text

Your task:
- Extract ONLY the representation body.
- DO NOT remove any text, sentences, or paragraphs within the representation body.
- DO NOT remove section headers (e.g., "Visual Impact", "Noise") that are part of the body.
- Preserve the original wording exactly.
- DO NOT paraphrase.
- DO NOT correct spelling.

Remove:
- administrative headers
- representation titles or labels (e.g. "Representation 001 Objection")
- names, addresses, contact details appearing outside representation body
- transmission markers (e.g. "BY EMAIL")
- salutations
- signatures
- sign-offs
- office addresses
-  ppendices or other supplementary attachments

Important rules:
- The output should begin with the first sentence of the actual representation body, not with a title or label.
- The output should end with the last sentence of the actual representation body, before any signature, name, or address block.
- If an attachment contains the substantive representation itself, extract its representation body.
- Keep argument summaries only if they contain actual reasoning.
- Output only the cleaned representation body text and nothing else.
""".strip()

def load_document(txt_path: Path):
    with open(txt_path, "r", encoding="utf-8") as f:
        text = f.read().strip()
    return text


#optimizing context length to increase processing speed
def choose_num_ctx(text):
    words = len(text.split())

    if words <= 2600:
        return 8192
    elif words <= 4000:
        return 12288
    elif words <= 6200:
        return 16384
    else:
        return 18432


def run_ollama_chat(system_prompt, user_content, num_ctx):
    payload = {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_content}
        ],
        "stream": False,
        "keep_alive": "30m",
        "options": {
            "temperature": 0,
            "repeat_penalty": 1.0,
            "seed":42,
            "num_ctx": num_ctx,
            "top_k": 1,
            "num_predict": 12288}} ## should be ok


    response = requests.post(OLLAMA_URL, json=payload, timeout=TIMEOUT)
    response.raise_for_status()
    data = response.json()
    return data["message"]["content"].strip()

all_txt_files = list(BASE_DIR.rglob("*.txt"))

if not all_txt_files:
    print(f"Error: Could not find any files in {BASE_DIR}")
else:
    files_to_process = sorted(all_txt_files)
    num_files = len(files_to_process)

    print(f"Found {num_files} total files. Processing all files...\n")

#replicating the folder structure
    for index, file_path in enumerate(files_to_process, 1):
        relative_path = file_path.relative_to(BASE_DIR)

        output_subdir = OUTPUT_DIR / relative_path.parent
        output_subdir.mkdir(parents=True, exist_ok=True)

        output_file = output_subdir / f"extracted_{file_path.stem}.txt"

        if output_file.exists():
            print(f"[{index}/{num_files}] Skipping already processed: {relative_path}")
            continue

        print(f"[{index}/{num_files}] Processing: {relative_path}")

        try:
            txt_text = load_document(file_path)
            word_count = len(txt_text.split())

            user_content = (
                "Extract the substantive representation text from this document.\n\n"
                "Input document:\n\n"
                f"{txt_text}")

            num_ctx = choose_num_ctx(txt_text)
            print(f"Words: {word_count}, num_ctx: {num_ctx}")

            output = run_ollama_chat(SYSTEM_PROMPT, user_content, num_ctx)

            with open(output_file, "w", encoding="utf-8") as f:
                f.write(output)

            print(f"Saved to {output_file}\n")

        except requests.exceptions.Timeout:
            print("Request timed out.\n")
        except requests.exceptions.HTTPError as e:
            print(f"HTTP error: {e}\n")
        except Exception as e:
            print(f"Unexpected error: {e}\n")

print("\nProcessing completed.")

Found 2282 total files. Processing all files...

[1/2282] Processing: Aberdeen_City_210665_DPP/210665_DPP-Objects_-_Ken_Cumming__Veripos_-2029057.txt
Words: 349, num_ctx: 8192
Saved to /mnt/clips_home/llm_cleanup_new_code/NO_STRUCTURE_LLM_CLEANED/Aberdeen_City_210665_DPP/extracted_210665_DPP-Objects_-_Ken_Cumming__Veripos_-2029057.txt

[2/2282] Processing: Aberdeen_City_220026_DPP/220026_DPP-Objects_-_Brodies_LLP__on_Behalf_Of_Helix_Well_Ops__UK__Ltd_-2109948.txt
Words: 4750, num_ctx: 16384
Saved to /mnt/clips_home/llm_cleanup_new_code/NO_STRUCTURE_LLM_CLEANED/Aberdeen_City_220026_DPP/extracted_220026_DPP-Objects_-_Brodies_LLP__on_Behalf_Of_Helix_Well_Ops__UK__Ltd_-2109948.txt

[3/2282] Processing: Aberdeen_City_231134_DPP/231134_DPP-Objects-Kathryn_Duncan-2288359.txt
Words: 398, num_ctx: 8192
Saved to /mnt/clips_home/llm_cleanup_new_code/NO_STRUCTURE_LLM_CLEANED/Aberdeen_City_231134_DPP/extracted_231134_DPP-Objects-Kathryn_Duncan-2288359.txt

[4/2282] Processing: Aberdeen_City_231134_